# Creation of an Interactive Map showing School Locations within NI and the distance from the Users location to each School

## STEP 1: Importing Required Libraries 

Step 1 enables the set up of the tools required for this project.  Pandas are imported to allow CSV files to be loaded and manipulated.  Folium is imported to enable to use of an interactive map background.  Geodesic is imported to enable distance calculations.

In [1]:
import pandas as pd
import folium
from geopy.distance import geodesic

C:\Users\rjwil\anaconda3\envs\schools_env\lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


## STEP 2: Data Loading

Step 2 loads the dataset to be utilised within the study into the Notebook using the  pandas read_csv() function.  This allows the dataset to be further manipulated.  The head function allowed the first few rows of the dataset to be displayed ensuring that it was correct and the headings were known. 

In [2]:
df = pd.read_csv("data/schools.csv")
df.head()

,Reference,Institution_Name,Address_1,Postcode,Email,Institution_Type,Latitude,Longitude,Pupil_Teacher_Ratio,Number_Pupils_2015
0,1AB0427,174 Trust Playgroup,Duncairn Complex,BT14 6BP,kaokane@outlook.com,VP Pre-schools,54.61253,-5.93645,NaN,NaN
1,5420059,Abbey Christian Brothers Grammar School,77a Ashgrove Road,BT34 2QN,info@abbey.newry.ni.sch.uk,Secondary (grammar) school,54.19349,-6.32846,15.81,902.0
2,3210313,Abbey Community College,Bridge Road,BT37 0EA,info@abbeycommunitycollege.newtownabbey.ni.sch.uk,Secondary (non-grammar) school,54.69015,-5.91848,13.50,613.0
3,4016399,Abbey Primary School,90 MOVILLA ROAD,BT23 8RQ,info@abbeyprimary.newtownards.ni.sch.uk,Primary school,54.59601,-5.66723,24.83,550.0
4,3010862,Abbots Cross Primary School,86 DOAGH ROAD,BT37 9QW,info@abbotscrossps.newtownabbey.ni.sch.uk,Primary school,54.66801,-5.91408,19.06,305.0


## STEP 3: Preparation of Data

Step 3 utilises the dropna() tool to remove any rows that have latitude or longitude values missing.  This reduces the amount of errors within the points plotted on the map.  

In [3]:
 df = df.dropna(subset=["Latitude", "Longitude"])

## STEP 4: Defining Users Location

Within the creation of this interactive map the distance from the users location to each school is calculated.  Hence, the users location needs to be defined.

The user location is defined in the format of latitude, longitude co-ordinates.

The generic user location for this study is Portglenone, but these co-ordinates can be altered by each user. 

In [4]:
user_location = (54.87, -6.47) 

## STEP 5: Creation of a Folium Map

Step 5 creates a base map centered on the users location and adds a marker to outline the users locations visually, with it displaying the map. 

In [5]:
m = folium.Map(location=user_location, zoom_start=8)

folium.Marker(
    location=user_location,
    popup="Your Location",
    icon=folium.Icon(color="darkblue", icon="user")
).add_to(m)

m

## STEP 6: Identifying Data Categories

To outline all the types of school institutions within with dataset without going through manually, the unique values tool was utilised.  This informs the code within the next step and allows it to be accurate.

In [6]:
df["Institution_Type"].unique()

array(['VP Pre-schools', 'Secondary (grammar) school',
       'Secondary (non-grammar) school', 'Primary school',
       'Special school', 'Nursery school', 'Preparatory Schools',
       'Peripatetic'], dtype=object)

## STEP 7: Filter Creation and Classification

Step 7 creates 6 seperate map layers using folium feature group tool, whcih allows the user to toggle the visibility of each layer on the map.  A dictionary was created to link each institution type to a layer and assign a colour for representation on the map.

In [7]:
primary_layer = folium.FeatureGroup(name="Primary Schools")
non_grammar_layer = folium.FeatureGroup(name="Secondary Non-Grammar Schools")
grammar_layer = folium.FeatureGroup(name="Secondary Grammar Schools")
nursery_layer = folium.FeatureGroup(name="Nursery Schools")
special_layer = folium.FeatureGroup(name="Special Schools")
pre_layer = folium.FeatureGroup(name="Pre-Schools")

groups = {
    "Primary school": {"layer": primary_layer, "color": "blue"},
    "Secondary (non-grammar) school": {"layer": non_grammar_layer, "color": "red"},
    "Secondary (grammar) school": {"layer": grammar_layer, "color": "green"},
    "Nursery school": {"layer": nursery_layer, "color": "orange"},
    "Special school": {"layer": special_layer, "color": "purple"},
    "VP Pre-schools": {"layer": pre_layer, "color": "pink"},
    "Preparatory Schools": {"layer": pre_layer, "color": "pink"}
}

## STEP 8: Adding Pointers for School Locations and Calculation of Distance

Step 8 uses the loop tool to go through each row of the dataset and identify the co-ordinates of the school, identifying its location and then calculating the distance between each school and the users location (defined in step 4).  Additonally, a pop-up for each school showing the institution name, institution type and distance between the school and user is created.  Lastly, each school point is assigned to the correct layer as defined in step 7 and added to the map.  To reduce errors if a school type that h=has not been listed within the layers is identified, the loop skips to the next row and continues the process.

In [8]:
for _, row in df.iterrows():
    
    school_location = (row["Latitude"], row["Longitude"])
    distance = geodesic(user_location, school_location).km
    
    popup_text = f"{row['Institution_Name']}<br>Type: {row['Institution_Type']}<br>Distance: {distance:.2f} km"
    
    school_type = str(row["Institution_Type"]).strip()

    if school_type in groups:
        
        group = groups[school_type]["layer"]
        color = groups[school_type]["color"]
        
        folium.CircleMarker(
            location=school_location,
            radius=3.5,
            color=color,
            fill=True,
            popup=popup_text
        ).add_to(group)

    else:
        continue

## STEP 9: Adding Layers Map and Enabling Filtering

Step 9 adds each feature group layer to the map and a layer control was added to allow the layers to be filtered on the map.

In [9]:
primary_layer.add_to(m)
non_grammar_layer.add_to(m)
grammar_layer.add_to(m)
nursery_layer.add_to(m)
special_layer.add_to(m)
pre_layer.add_to(m)

folium.LayerControl().add_to(m)

m

## STEP 10: Addition of a 5km Buffer

Step 10 uses the folium circle tool to add a 5km circle radius around the users location, enabling a visual of the schools within this distance.  The distance can be altered by future users.

In [10]:
folium.Circle(
    location=user_location,
    radius=5000,            #Radius Distance in meters, therefore 5000 represents 5km.  This can be altered by the user.
    color='black',
    fill=False,          
    fill_opacity=0,      
    weight=1
).add_to(m)

m

## STEP 11: Saving Map as an HTML

The final step is to save the map as an HTML file so it can be repeatedly accessed.

In [11]:
m.save("schools_map.html")